# OOHScout — F2 Adaptation: IH-35 Centerline for McLennan County

**Current scope:** F2 — Base Geometry (IH-35 highway centerline)

This notebook fetches every OSM way tagged `highway=motorway, ref=I 35` inside the McLennan County boundary (from F1), filters to LineStrings only, projects to EPSG:32614, and caches the result as a GeoPackage. It does not sample candidate sites yet (that's F7) or compute a buffer (F6).

**Sources:** OpenStreetMap Overpass API via OSMnx. Boundary from F1's cached study area.

**Production module:** [`backend/src/oohscout/track_a_spatial/corridor.py`](../../../../backend/src/oohscout/track_a_spatial/corridor.py). The notebook is the learning artifact; the module is what FastAPI + agent tools import.

## 1. Imports and study-area load

F2 depends on F1's boundary — no new geocoding call is made here.

In [ ]:
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt

from oohscout.track_a_spatial import (
    load_or_build_study_area,
    load_or_build_ih35_centerline,
)

print(f'geopandas {gpd.__version__}')

In [ ]:
# Repo-root detection so this notebook works whether Jupyter launches from
# the repo root or from inside the chapter folder.
working_dir = Path.cwd().resolve()
repo_candidates = [working_dir, *working_dir.parents]
REPO_ROOT = next(
    (p for p in repo_candidates if (p / 'pyproject.toml').exists()),
    None,
)
assert REPO_ROOT is not None, 'Run this notebook from inside the repository.'

DATA_DIR = REPO_ROOT / 'backend' / 'data' / 'processed'
print(f'DATA_DIR: {DATA_DIR}')

PLACE = 'McLennan County, Texas'
CRS_METRIC = 32614

In [ ]:
# F1 boundary — cache-first, no network call on repeat.
study = load_or_build_study_area(
    place=PLACE,
    crs_metric=CRS_METRIC,
    cache_dir=DATA_DIR,
    reference_total_area_km2=2746.0,
)
print(f'McLennan area: {study.study_area.area / 1e6:,.1f} km²')

## 2. Fetch IH-35 inside McLennan

Same F1 discipline: cache-first, assertions raise, projected metric CRS owns measurement.

The production function does five things:

1. Query Overpass for every `highway=motorway` feature inside `admin_poly`.
2. Keep LineString / MultiLineString only.
3. Keep rows whose `ref` starts with `I 35` (also matches multi-refs like `I 35;US 77`).
4. Dedupe on `osmid` — OSMnx occasionally returns the same way twice with identical geometry.
5. Project to EPSG:32614 and cache as `mclennan_ih35_centerline.gpkg`.

In [ ]:
corridor = load_or_build_ih35_centerline(
    admin_poly=study.admin_poly,
    cache_dir=DATA_DIR,
    crs_metric=CRS_METRIC,
)

print(f'segments        : {len(corridor.corridor_gdf)}')
print(f'geom types      : {sorted(corridor.corridor_gdf.geometry.geom_type.unique())}')
print(f'osmid unique    : {corridor.corridor_gdf["osmid"].is_unique}')
print(f'total length km : {corridor.total_length_m / 1000:,.2f}')
print(f'cache           : {corridor.cache_path.name}')

## 3. Visual check

The red lines should trace IH-35 diagonally through McLennan County, passing through Waco. Both directions of travel are drawn — that's why the total-length number is ~130 km rather than the ~55 km Google Maps reports for a single direction.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# County boundary as context
study.admin_gdf_metric.plot(
    ax=ax, facecolor='none', edgecolor='#0a84ff', linewidth=1.5, alpha=0.8
)
# IH-35 in red
corridor.corridor_gdf_metric.plot(
    ax=ax, color='#ff2d55', linewidth=1.5
)
cx.add_basemap(
    ax, crs=corridor.corridor_gdf_metric.crs, source=cx.providers.Esri.WorldImagery
)
ax.set_title(f'IH-35 through {PLACE} — F2', fontsize=13)
ax.set_aspect('equal')
ax.axis('off')
plt.tight_layout()

check_png = DATA_DIR / 'mclennan_ih35_f2_check.png'
fig.savefig(check_png, dpi=120)
print(f'Saved boundary check image to {check_png}')

plt.show()

## F2 completion gate

F2 ships when:

1. Segments returned > 0 (Overpass had data)
2. Every geometry is `LineString` or `MultiLineString` (no accidental Point / Polygon)
3. `osmid` is unique after dedupe (safe for downstream joins to permits / AADT / candidates)
4. Total length is a plausible IH-35-through-McLennan number (~100-160 km, includes both directions)
5. Cache file written to `backend/data/processed/mclennan_ih35_centerline.gpkg`
6. Red lines visually trace IH-35 through Waco on the Esri basemap

The pytest at `backend/tests/track_a/test_corridor.py` enforces items 1-5 automatically. Item 6 stays a human eye check for the notebook.